In [23]:
from recbole.quick_start import run_recbole

from itertools import product
from pathlib import Path

import pandas as pd
import json

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

CONFIG_ROOT = PROJECT_ROOT / "recbole" / "config"

RESULTS_DIR = PROJECT_ROOT / "tuning_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [25]:
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_seed


def evaluate_traditional_model(model_name, config_path, params):
    """Fit a traditional RecBole model and evaluate on validation data."""

    config = Config(
        model=model_name,
        dataset="ml-100k",
        config_file_list=[str(config_path)],
        config_dict={
            "seed": 2020,
            "reproducibility": True,
            "use_gpu": False,
            "data_path": str(PROJECT_ROOT / "dataset"),
            **params,
        },
    )

    init_seed(config["seed"], config["reproducibility"])

    dataset = create_dataset(config)
    train_data, valid_data, _ = data_preparation(config, dataset)

    # EASE and SLIMElastic fit their parameters during initialization
    model = get_model(model_name)(
        config,
        train_data.dataset
    ).to(config["device"])

    trainer = get_trainer(
        config["MODEL_TYPE"],
        config["model"]
    )(config, model)

    valid_result = trainer.evaluate(
        valid_data,
        load_best_model=False
    )

    return {
        "best_valid_score": valid_result["mrr@10"],
        "best_valid_result": valid_result,
    }

In [22]:
def tune_model(
    name,
    model,
    config_path,
    search_space,
    fixed_params=None,
    save=True
):
    """
    Exhaustively tune a RecBole model.

    Parameters
    ----------
    name : str
        Name used for displaying/saving results.

    model : str
        RecBole model name.

    config_path : str or Path
        Path to the model's YAML configuration.

    search_space : dict
        Hyperparameters and values to test.

        Example:
        {
            "k": [20, 50, 100],
            "shrink": [0, 10, 50]
        }

    fixed_params : dict, optional
        Parameters that should remain fixed.
        Useful for things such as knn_method.

    save : bool
        Save tuning results to CSV.

    Returns
    -------
    DataFrame
        All tuning experiments.
    """

    if fixed_params is None:
        fixed_params = {}

    # Parameter names
    param_names = list(search_space.keys())

    # All combinations
    combinations = list(
        product(*search_space.values())
    )

    results = []

    print(f"Tuning {name}")
    print(f"Number of experiments: {len(combinations)}")
    print("=" * 60)

    for i, values in enumerate(combinations, start=1):

        params = dict(zip(param_names, values))

        config_dict = {
            "model": model,
            "dataset": "ml-100k",
            **fixed_params,
            **params,
        }

        print(
            f"[{i}/{len(combinations)}] "
            f"{name}: {params}"
        )

        if model in {"EASE", "SLIMElastic"}:
            result = evaluate_traditional_model(
                model_name=model,
                config_path=config_path,
                params=params,
            )
        else:
            result = run_recbole(
                model=model,
                dataset="ml-100k",
                config_dict=config_dict,
                config_file_list=[str(config_path)],
                saved=False,
            )

        valid = result["best_valid_result"]

        row = {
            **params,
            "valid_score": result["best_valid_score"],
            **valid,
        }

        valid = result["best_valid_result"]

        row = {
            **params,
            "valid_score": result["best_valid_score"],
            **valid
        }

        results.append(row)

    df = pd.DataFrame(results)

    # Best first
    df = df.sort_values(
        "valid_score",
        ascending=False
    ).reset_index(drop=True)

    if save:
        output_path = RESULTS_DIR / f"{name}_tuning.csv"
        df.to_csv(output_path, index=False)

        print(f"\nSaved to:")
        print(output_path)

    print("\nBest configuration:")
    display(df.head(1))

    return df

In [11]:
itemcf_space = {
    "k": [20, 50, 100, 200],
    "shrink": [0, 10, 50, 100]
}

In [26]:
itemcf_results = tune_model(
    name="ItemCF",
    model="ItemKNN",
    config_path=CONFIG_ROOT / "ItemKNN" / "ml-100k.yaml",
    search_space=itemcf_space,
    fixed_params={
        "knn_method": "item"
    }
)

Tuning ItemCF
Number of experiments: 16
[1/16] ItemCF: {'k': 20, 'shrink': 0}


13 Oct 22:09    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v378a84a4e8ede30b35db7597ce0cc5982cb81449c.json']
13 Oct 22:09    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm =

[2/16] ItemCF: {'k': 20, 'shrink': 10}


13 Oct 22:10    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v378a84a4e8ede30b35db7597ce0cc5982cb81449c.json']
13 Oct 22:10    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm =

[3/16] ItemCF: {'k': 20, 'shrink': 50}


...

13 Oct 22:11    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v378a84a4e8ede30b35db7597ce0cc5982cb81449c.json']
13 Oct 22:11    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm =

[16/16] ItemCF: {'k': 200, 'shrink': 100}


13 Oct 22:12    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v378a84a4e8ede30b35db7597ce0cc5982cb81449c.json']
13 Oct 22:12    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm =


Saved to:
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/tuning_results/ItemCF_tuning.csv

Best configuration:


,k,shrink,valid_score,recall@10,mrr@10,ndcg@10,hit@10,precision@10
0,100,100,0.3869,0.2007,0.3869,0.2216,0.7264,0.1496


In [13]:
usercf_results = tune_model(
    name="UserCF",
    model="ItemKNN",
    config_path=CONFIG_ROOT / "ItemKNN" / "ml-100k.yaml",
    search_space=itemcf_space,
    fixed_params={
        "knn_method": "user"
    }
)

Tuning UserCF
Number of experiments: 16
[1/16] UserCF: {'k': 20, 'shrink': 0}


13 Oct 21:51    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:51    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
2026-10-13 21:51:35,265	WARNING session.py:91 -- Session not detected. You should not be calling `report` outside `tuner.fit()` or while using the class API. 
2026-10-13 21:51:35,265	WARNING session.py:97 --   File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line

[2/16] UserCF: {'k': 20, 'shrink': 10}


13 Oct 21:51    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:51    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)


[3/16] UserCF: {'k': 20, 'shrink': 50}


...

13 Oct 21:51    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:51    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)


[16/16] UserCF: {'k': 200, 'shrink': 100}


13 Oct 21:51    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:51    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)



Saved to:
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/tuning_results/UserCF_tuning.csv

Best configuration:


,k,shrink,valid_score,recall@10,mrr@10,ndcg@10,hit@10,precision@10
0,50,0,0.401,0.216,0.401,0.2355,0.755,0.1562


In [ ]:
bpr_space = {
    "embedding_size": [16, 32, 64, 128],
    "learning_rate": [0.0001, 0.001, 0.01],
}

In [15]:
bpr_results = tune_model(
    name="BPR",
    model="BPR",
    config_path=CONFIG_ROOT / "BPR" / "ml-100k.yaml",
    search_space=bpr_space
)

Tuning BPR
Number of experiments: 36
[1/36] BPR: {'embedding_size': 16, 'learning_rate': 0.0001, 'reg_weight': 0.0}


13 Oct 21:52    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:52    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
2026-10-13 21:52:14,158	WARNING session.py:91 -- Session not detected. You should not be calling `report` outside `tuner.fit()` or while using the class API. 
2026-10-13 21:52:14,158	WARNING session.py:97 --   File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line

[2/36] BPR: {'embedding_size': 16, 'learning_rate': 0.0001, 'reg_weight': 0.0001}


13 Oct 21:52    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:52    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)


[3/36] BPR: {'embedding_size': 16, 'learning_rate': 0.0001, 'reg_weight': 0.001}


...

13 Oct 21:54    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:54    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)


[36/36] BPR: {'embedding_size': 128, 'learning_rate': 0.01, 'reg_weight': 0.001}


13 Oct 21:54    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:54    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)



Saved to:
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/tuning_results/BPR_tuning.csv

Best configuration:


,embedding_size,learning_rate,reg_weight,valid_score,recall@10,mrr@10,ndcg@10,hit@10,precision@10
0,32,0.01,0.0,0.3581,0.1904,0.3581,0.2038,0.7169,0.1393


In [16]:
slim_space = {
    "alpha": [0.0001, 0.001, 0.01, 0.1],
    "l1_ratio": [0.1, 0.5, 0.9]
}

slim_results = tune_model(
    name="SLIM",
    model="SLIMElastic",
    config_path=CONFIG_ROOT / "SLIMElastic" / "ml-100k.yaml",
    search_space=slim_space
)

Tuning SLIM
Number of experiments: 12
[1/12] SLIM: {'alpha': 0.0001, 'l1_ratio': 0.1}


13 Oct 21:54    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:54    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


[2/12] SLIM: {'alpha': 0.0001, 'l1_ratio': 0.5}


13 Oct 21:54    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:54    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


[3/12] SLIM: {'alpha': 0.0001, 'l1_ratio': 0.9}


...

13 Oct 21:56    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:56    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


[12/12] SLIM: {'alpha': 0.1, 'l1_ratio': 0.9}


13 Oct 21:56    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:56    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]



Saved to:
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/tuning_results/SLIM_tuning.csv

Best configuration:


,alpha,l1_ratio,valid_score,recall@10,mrr@10,ndcg@10,hit@10,precision@10
0,0.1,0.1,0.4165,0.2271,0.4165,0.2495,0.7625,0.1709


In [17]:
ease_space = {
    "reg_weight": [10, 50, 100, 200, 500, 1000]
}

ease_results = tune_model(
    name="EASE",
    model="EASE",
    config_path=CONFIG_ROOT / "EASE" / "ml-100k.yaml",
    search_space=ease_space
)

Tuning EASE
Number of experiments: 6
[1/6] EASE: {'reg_weight': 10}


13 Oct 21:57    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:57    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


[2/6] EASE: {'reg_weight': 50}


13 Oct 21:57    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:57    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


[3/6] EASE: {'reg_weight': 100}


...

13 Oct 21:57    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:57    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


[6/6] EASE: {'reg_weight': 1000}


13 Oct 21:57    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
13 Oct 21:57    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]



Saved to:
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/tuning_results/EASE_tuning.csv

Best configuration:


,reg_weight,valid_score,recall@10,mrr@10,ndcg@10,hit@10,precision@10
0,500,0.4189,0.2386,0.4189,0.2568,0.7815,0.1766


In [28]:
fism_space = {
    "embedding_size": [32, 64],
    "learning_rate": [0.001, 0.005],
    "alpha": [0.5],
    "reg_weights": [
        [0.0001, 0.0001],
        [0.001, 0.001],
    ],
}

fism_results = tune_model(
    name="FISM",
    model="FISM",
    config_path=CONFIG_ROOT / "FISM" / "ml-100k.yaml",
    search_space=fism_space,
)

Tuning FISM
Number of experiments: 8
[1/8] FISM: {'embedding_size': 32, 'learning_rate': 0.001, 'alpha': 0.5, 'reg_weights': [0.0001, 0.0001]}


14 Oct 08:13    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v378a84a4e8ede30b35db7597ce0cc5982cb81449c.json']
14 Oct 08:13    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 10
stopping_step = 20
clip_grad_norm =

[2/8] FISM: {'embedding_size': 32, 'learning_rate': 0.001, 'alpha': 0.5, 'reg_weights': [0.001, 0.001]}


14 Oct 08:17    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v378a84a4e8ede30b35db7597ce0cc5982cb81449c.json']
14 Oct 08:17    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 10
stopping_step = 20
clip_grad_norm =

[3/8] FISM: {'embedding_size': 32, 'learning_rate': 0.005, 'alpha': 0.5, 'reg_weights': [0.0001, 0.0001]}


...

14 Oct 08:46    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v378a84a4e8ede30b35db7597ce0cc5982cb81449c.json']
14 Oct 08:46    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 2048
learner = adam
learning_rate = 0.005
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 10
stopping_step = 20
clip_grad_norm =

[8/8] FISM: {'embedding_size': 64, 'learning_rate': 0.005, 'alpha': 0.5, 'reg_weights': [0.001, 0.001]}


14 Oct 08:52    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v378a84a4e8ede30b35db7597ce0cc5982cb81449c.json']
14 Oct 08:52    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 2048
learner = adam
learning_rate = 0.005
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 10
stopping_step = 20
clip_grad_norm =


Saved to:
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/tuning_results/FISM_tuning.csv

Best configuration:


,embedding_size,learning_rate,alpha,reg_weights,valid_score,recall@10,mrr@10,ndcg@10,hit@10,precision@10
0,64,0.005,0.5,"[0.0001, 0.0001]",0.3429,0.1821,0.3429,0.1976,0.7169,0.1417
